# Go2 + Franka Pick-and-Carry

Genesis で、Go2 の背中に固定した Franka が立方体を目標へ運びます。

**先にランタイムを GPU にしてください。** メニュー「ランタイム」→「ランタイムのタイプを変更」→ ハードウェア アクセラレータを **T4 GPU** または **L4 GPU** にします。CPU のままだと Genesis は `gs.cpu` に落ち、地形の準備で止まります。変更したあとは「ランタイムを再起動」して、上のセルからやり直します。

Colab には 3D ウィンドウがありません。ロボットは mp4 をこのノートの中で再生します。ビューアは使いません。

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

os.environ['PYOPENGL_PLATFORM'] = 'egl'

if shutil.which('nvidia-smi') is None:
    raise SystemExit(
        'このランタイムは CPU です。GPU は使われていません。\n'
        '「ランタイム」→「ランタイムのタイプを変更」→ ハードウェア アクセラレータを GPU にして、\n'
        'セッションを再起動してから、このセルを再実行してください。'
    )
subprocess.check_call(['nvidia-smi'])

candidates = [
    Path('/content/PAI_Task'),
    Path('/content/drive/MyDrive/PAI_Task'),
    Path('/content/drive/MyDrive/content/PAI_Task'),
    Path.cwd(),
]
root = next((path for path in candidates if (path / 'pick_and_carry').is_dir()), None)
if root is None and Path('/content').exists() and not Path('/content/drive').exists():
    from google.colab import drive
    drive.mount('/content/drive')
    root = next((path for path in candidates if (path / 'pick_and_carry').is_dir()), None)
if root is None:
    raise SystemExit('pick_and_carry が見つかりません。プロジェクトを /content/PAI_Task かドライブに置いてください。')
os.chdir(root)
print('cwd', Path.cwd())

In [ ]:
!pip install -q -r requirements.txt
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit(
        'PyTorch から GPU が見えません。ランタイムを GPU にしてセッションを再起動してください。\n'
        'CPU 版の torch を入れた場合も、同じ再起動が必要です。'
    )
print('device', torch.cuda.get_device_name(0))

In [ ]:
from pick_and_carry.config import NUM_ACTIONS, NUM_OBS, OBS_LAYOUT, default_urdf_path, layout_slices
from pick_and_carry.urdf_tools import validate_urdf

print('obs', NUM_OBS, 'act', NUM_ACTIONS)
for name, sl in layout_slices(OBS_LAYOUT).items():
    print(f'{sl.start:02d}:{sl.stop:02d}  {name}')
report = validate_urdf(default_urdf_path(), require_actuated_names=True)
print(report['path'])
print('mount', report['mount_parent'], '->', report['mount_child'], report['mount_xyz'])

## 公式 URDF を使うとき

`go2.urdf` と `panda.urdf` を置いたときだけ結合します。無ければプリミティブの形で動きます。アームが重くて転ぶ場合は質量スケール `0.25` のままにします。

In [ ]:
import subprocess
from pathlib import Path
go2 = Path('go2.urdf')
panda = Path('panda.urdf')
if go2.is_file() and panda.is_file():
    subprocess.check_call(['python', 'scripts/merge_urdf.py', '--go2', 'go2.urdf', '--panda', 'panda.urdf', '--arm-mass-scale', '0.25'])
else:
    print('go2.urdf / panda.urdf が無いのでプリミティブ URDF を使います')

## 描画の確認

1 体だけを 4 秒間録画し、下に再生します。初回はカーネルのコンパイルで数分かかります。3D ウィンドウは開きません。

In [ ]:
import torch
from IPython.display import Video, display
from pick_and_carry.config import EnvConfig
from pick_and_carry.env import PickAndCarryEnv

if not torch.cuda.is_available():
    raise SystemExit('GPU がありません。ランタイムを GPU にして再起動してください。')

preview = 'logs/colab_preview.mp4'
env = PickAndCarryEnv(cfg=EnvConfig(
    num_envs=1,
    backend='gpu',
    show_viewer=False,
    record_path=preview,
    scripted_gait=True,
    seed=0,
))
print('genesis device', env.device)
if 'cpu' in str(env.device).lower():
    env.close()
    raise SystemExit('Genesis が CPU に落ちました。GPU ランタイムでセッションを再起動してください。')
obs = env.reset()
action = torch.zeros(env.num_envs, env.num_actions, device=obs.device)
for _ in range(200):
    obs, _, _, _ = env.step(action)
env.close()
print('video', preview)
display(Video(preview, embed=True, width=960))

## 短い PPO

GPU が見えているときだけ学習します。T4 ではまず 8 環境です。通ったら `--num-envs 32` に上げられます。チェックポイントは `logs/ppo` に残ります。

In [ ]:
import shutil
import subprocess
import sys
import torch

if shutil.which('nvidia-smi') is None or not torch.cuda.is_available():
    raise SystemExit('GPU がありません。CPU のまま学習すると地形の準備で止まります。ランタイムを GPU にして再起動してください。')
subprocess.check_call([
    sys.executable, '-m', 'pick_and_carry.train_ppo',
    '--backend', 'gpu',
    '--num-envs', '8',
    '--iterations', '5',
    '--num-steps', '16',
    '--save-interval', '5',
    '--log-dir', 'logs/ppo',
])

## 学習結果を録画して再生

保存した重みを 1 体で 8 秒再生し、mp4 を表示します。

In [ ]:
import subprocess
import sys
from IPython.display import Video, display

playback = 'logs/colab_ppo.mp4'
subprocess.check_call([
    sys.executable, '-m', 'pick_and_carry.train_ppo',
    '--play', '--resume', 'latest', '--backend', 'gpu',
    '--num-envs', '1', '--seconds', '8', '--record', playback,
    '--log-dir', 'logs/ppo',
])
display(Video(playback, embed=True, width=960))